# box-array-to-tensor-with-recipe composite — cx8: boxing gated by the global grad-tracking toggle

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 3 atoms together: `box-array-to-tensor-with-recipe`, `recipe-dataclass`, `grad-tracking-global-toggle`
> Running the final beacon reports progress against all 3 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "box-array-to-tensor-with-recipe"
DD_ATOM_IDS = ["box-array-to-tensor-with-recipe", "recipe-dataclass", "grad-tracking-global-toggle"]
DD_SUBTOPICS = ["Backprop: Box array as Tensor + recipe", "Backprop: Recipe dataclass", "Backprop: Grad-tracking toggle"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Boxing gated by the global grad toggle

When `grad_tracking_enabled` is `False`, the wrapper short-circuits: the output Tensor still gets created, but `requires_grad=False` and **no Recipe** is attached. That's how `no_grad` saves bookkeeping.

When it's `True` and any input has `requires_grad=True`, the wrapper boxes the raw output AND attaches a freshly-constructed 4-field Recipe.

Two atoms wired through one decision:
- `grad-tracking-global-toggle` decides IF a Recipe is built.
- `recipe-dataclass` defines WHAT shape the Recipe takes.
- `box-array-to-tensor-with-recipe` performs the wrap + conditional attach.

### Composite Exercise — boxing gated by the global grad-tracking toggle

**Atoms exercised together**: `box-array-to-tensor-with-recipe`, `recipe-dataclass`, `grad-tracking-global-toggle`

Implement `cx8_box_gated(out_raw, fwd_fn, raw_args, kwargs, parents, any_input_requires_grad)` — the second half of `wrap_forward_fn`, but with the global toggle inline.

Read the module-level `grad_tracking_enabled` (don't snapshot it into a closure — read it via `globals()['grad_tracking_enabled']` so the latest value wins).

Compute `requires_grad = grad_tracking_enabled AND any_input_requires_grad`. Then:

- Box: `out = MiniTensor(out_raw, requires_grad=requires_grad)` (ALWAYS — the caller always expects a MiniTensor back).
- Recipe IFF `requires_grad`: `out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)`.

Return `out`. Identity matters — `out.array is out_raw` (no copy).

In [ ]:
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx8_box_gated(out_raw, fwd_fn, raw_args, kwargs, parents, any_input_requires_grad):
    # Read the global through globals() — never snapshot into a closure.
    toggle = globals()['grad_tracking_enabled']
    requires_grad = bool(toggle and any_input_requires_grad)
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return out


<details><summary>Show solution — cx8</summary>

```python
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def cx8_box_gated(out_raw, fwd_fn, raw_args, kwargs, parents, any_input_requires_grad):
    # Read the global through globals() — never snapshot into a closure.
    toggle = globals()['grad_tracking_enabled']
    requires_grad = bool(toggle and any_input_requires_grad)
    out = MiniTensor(out_raw, requires_grad=requires_grad)
    if requires_grad:
        out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
    return out
```

**Why read the toggle through `globals()`.** A naive `def f(): return grad_tracking_enabled and ...` closes over the *binding*, but if the test cell rebinds the global between calls (which `no_grad` does on every enter/exit) you can stale-bind to the old value in some execution environments. Reading the dict on each call always sees the current binding.

**Why `if requires_grad` and not `if any_input_requires_grad`.** The toggle has to veto: even if an input wants gradients, a `no_grad` block means we discard the bookkeeping. The conditional reads the post-AND value, not the raw input flag.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 3 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx8',
        'subtopics': ["Backprop: Box array as Tensor + recipe", "Backprop: Recipe dataclass", "Backprop: Grad-tracking toggle"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()